# <b>The Skeleton Run

In [1]:
!pip install -q \
    transformers==4.45.2 \
    datasets==3.0.1 \
    peft==0.13.2 \
    trl==0.11.4 \
    accelerate==1.0.1 \
    safetensors==0.8.0 \
    tokenizers==0.20.3 \
    huggingface_hub==0.36.2 \
    numpy==1.26.4 \
    pandas==3.0.3 \
    tqdm==4.68.2

# 1. The SETUP

## 1. Importing Libraries

In [2]:
import os
import torch

from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
)

from peft import PeftModel

## 2. Device Configuration
The system automatically uses CUDA when a GPU is available; otherwise, it falls back to CPU.
This allows the same inference pipeline to run in both GPU and CPU environments.

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


## 3. Mount Google Drive
During development, trained Kaarshika models are loaded from Google Drive.
In the Flask deployment, these model paths will be replaced with the appropriate server/local model directories.

In [4]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Locate the models.

In [5]:
import os
BASE_DIR = "/content/drive/MyDrive/Kaarshika/models"

### 4.1 Locate the <b>INTENT ROUTER (BERT)</b> model.

In [6]:
INTENT_MODEL_PATH = f"{BASE_DIR}/classifier_lora"

### 4.2 Locate the <b>GENERATIVE</b> model.

In [7]:
GEN_MODEL_PATH = f"{BASE_DIR}/qwen2.5-3b-lora"

### 4.3 Locate the <b>RL Agent</b>.

In [8]:
RL_AGENT_PATH = f"{BASE_DIR}/reward_model"

## 5. Load <b>INTENT ROUTER

In [9]:
INTENT_BASE_MODEL = 'distilbert-base-uncased'

intent_tokenizer = AutoTokenizer.from_pretrained(INTENT_BASE_MODEL)
intent_base_model = AutoModelForSequenceClassification.from_pretrained(INTENT_BASE_MODEL, num_labels = 2)

intent_model = PeftModel.from_pretrained(
    intent_base_model,
    INTENT_MODEL_PATH
)

intent_model.to(device)
intent_model.eval()

print("Intent Router Loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Intent Router Loaded.


In [10]:
INTENT_LABELS = {
    0: "INFORMATION",
    1: "DECISION"
}

## 6. Load Generative LLM

In [11]:
GEN_BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_BASE_MODEL)
gen_base_model = AutoModelForCausalLM.from_pretrained(
    GEN_BASE_MODEL,
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map = 'auto'
)

gen_model = PeftModel.from_pretrained(
    gen_base_model,
    GEN_MODEL_PATH
)

gen_model.eval()

print("Generative LLM Loaded.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Generative LLM Loaded.


## 7. Load <b>Reward</b> Model

In [12]:
reward_tokenizer = AutoTokenizer.from_pretrained(RL_AGENT_PATH)
reward_base_model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels = 1)

reward_model = PeftModel.from_pretrained(
    reward_base_model,
    RL_AGENT_PATH
)

reward_model.eval()

print("Reward Model Loaded.")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Reward Model Loaded.


---

---

# 2. The HELPERS

## Classification by <b>BERT

In [13]:
def classify_intent(query):
    inputs = intent_tokenizer(query, return_tensors = 'pt', truncation = True, max_length = 512)

    with torch.no_grad():
        outputs = intent_model(**inputs)

    prediction = torch.argmax(outputs.logits, dim = -1).item()

    return INTENT_LABELS[prediction]

---

## <b>RAG</b> Pipeline

### Importing Libraries

In [18]:
!pip install -q faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 10.7 MB/s eta 0:00:0000:01


In [20]:
import faiss
from pypdf import PdfReader

from transformers import (DPRContextEncoder, DPRContextEncoderTokenizer,
                          DPRQuestionEncoder, DPRQuestionEncoderTokenizer,
                          AutoTokenizer, AutoModelForCausalLM)

### PDF Path

In [23]:
PDF_PATH = "/content/drive/MyDrive/Kaarshika/knowledge_base/kaarshika_KB.pdf"

In [24]:
reader = PdfReader(PDF_PATH)

print("Number of pages:", len(reader.pages))

Number of pages: 7


In [25]:
def read_and_split_pdf(filename):
    reader = PdfReader(filename)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + '\n'

    paragraphs = text.split("\n")

    return [
        para.strip()
        for para in paragraphs
        if len(para.strip()) > 0
    ]

In [28]:
paragraphs = read_and_split_pdf(PDF_PATH)
print("Number of paragraphs: ", len(paragraphs))

Number of paragraphs:  229


In [29]:
context_tokenizer = DPRContextEncoderTokenizer.from_pretrained('facebook/dpr-ctx_encoder-single-nq-base')
context_encoder = DPRContextEncoder.from_pretrained('facebook/dpr-ctx_encoder-single-nq-base')

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DPRQuestionEncoderTokenizer'. 
The class this function is called from is 'DPRContextEncoderTokenizer'.


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/dpr-ctx_encoder-single-nq-base were not used when initializing DPRContextEncoder: ['ctx_encoder.bert_model.pooler.dense.bias', 'ctx_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRContextEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRContextEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [30]:
def encode_contexts(text_list):
    embeddings = []

    for text in text_list:
        inputs = context_tokenizer(text, return_tensors = 'pt', padding = True, truncation = True, max_length = 1024)
        outputs = context_encoder(**inputs)
        embeddings.append(outputs.pooler_output)

    return torch.cat(embeddings).detach().numpy()

In [31]:
context_embeddings = encode_contexts(paragraphs)
context_embeddings

array([[-0.10561464,  0.10119364, -0.1116374 , ..., -0.7483823 ,
         0.09406368, -0.03221493],
       [-0.55419785,  0.24233595,  0.15689057, ...,  0.01826708,
         0.28466046,  0.5659051 ],
       [-0.01583152,  0.12472656, -0.22238368, ..., -0.8708607 ,
         0.19672264, -0.17679666],
       ...,
       [ 0.15029463, -0.20515345, -0.21841693, ..., -0.53023136,
         0.02470782,  0.48121828],
       [ 0.5097367 ,  0.2069647 , -0.2056292 , ..., -0.5687031 ,
         0.15142338, -0.19273692],
       [ 0.20511162,  0.34941053, -0.28636762, ..., -0.03687653,
         0.09214231,  0.29914021]], dtype=float32)

### FAISS Setup

In [32]:
import numpy as np

In [33]:
embedding_dim = 768
context_embeddings_np = np.array(context_embeddings).astype('float32')

Initialize L2 distance index and insert vectors

In [34]:
index = faiss.IndexFlatL2(embedding_dim)
index.add(context_embeddings_np)

### Load Question Encoder and Tokenizer

In [35]:
question_encoder = DPRQuestionEncoder.from_pretrained('facebook/dpr-question_encoder-single-nq-base')
question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained('facebook/dpr-question_encoder-single-nq-base')

config.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Search FAISS index for top-K matching paragraphs

In [36]:
def search_relevant_contexts(question, k = 5):
    question_inputs = question_tokenizer(question, return_tensors = 'pt')
    question_embedding = question_encoder(**question_inputs).pooler_output.detach().numpy()

    D, I = index.search(question_embedding, k)

    return D, I

In [43]:
query = "What are the water requirements of maize during flowering?"

D, I = search_relevant_contexts(query, k=5)

print("Distances:")
print(D)

print("\nIndices:")
print(I)

Distances:
[[76.20491  77.94055  78.51372  81.964584 82.84199 ]]

Indices:
[[203 109  33  88 146]]


In [44]:
for rank, idx in enumerate(I[0], start=1):

    print(f"\n===== Retrieved Context {rank} =====")
    print(paragraphs[idx])


===== Retrieved Context 1 =====
weeks before tuber harvest; stop irrigation just before pods mature (certain pulses/oilseeds).

===== Retrieved Context 2 =====
June–September; late July–October; summer Dec–Jan to March–April. Irrigation: Irrigate the field on the day of

===== Retrieved Context 3 =====
method preferred where water is available (seedlings ready ~1 week earlier). Age of seedlings: short-duration varieties 18

===== Retrieved Context 4 =====
Maize can be grown throughout the year at altitudes from sea level to about 300 m. Best rainfall range 600–900 mm.

===== Retrieved Context 5 =====
rainfall it grows well as rainfed crop; under prolonged dry periods after planting, irrigation can increase yield 150–200% over


In [46]:
def retrieve_guidance(question, k=5):
    D, I = search_relevant_contexts(question, k)
    retrieved_contexts = []

    for idx in I[0]:
        retrieved_contexts.append(paragraphs[idx])

    return retrieved_contexts

---

## The <b>RL Agent</b> Functions

### Score candidate actions

In [47]:
def score_actions(prompt, responses):
    scores = []

    for response in responses:
        text = f"{prompt}\nAction: {response}"
        inputs = reward_tokenizer(
            text, 
            return_tensors = 'pt',
            truncation = True,
            max_length = 512
        )

        inputs = {
            k: v.to(reward_model.device)
            for k, v in inputs.items()
        }

        with torch.no_grad():
            score = reward_model(**inputs).logits.item()

        scores.append(score)

    return scores

### Select best action

In [51]:
def select_best_action(responses, scores):
    best_index = scores.index(max(scores))

    return responses[best_index]

### Main <b>RL Agent</b>

In [52]:
def rl_agent(prompt, responses):
    scores = score_actions(
        prompt, responses
    )

    best_action = select_best_action(
        responses, scores
    )

    return best_action, scores

## Response Generator

---

# 3. The SKELETON